# 01 — WESAD features

First look at the windowed feature table built by `wesad_stress.features` and written to `data/processed/features.parquet`. The table holds one row per 60-second / 30-second-step window of chest signal, with 19 hand-rolled features (HRV, EDA tonic/phasic, EMG, respiration, accelerometer) and the window's majority condition label. See [`docs/features.md`](../docs/features.md) for the full column contract and the windowing rationale, and [`docs/schema.md`](../docs/schema.md) for the underlying raw-data contract.

This notebook loads the table, checks class balance per subject, and plots how the features distribute across the four conditions. The two figures are saved to `images/` at retina resolution so they can be referenced from `CHANGELOG.md`.


In [ ]:
%config InlineBackend.figure_format = 'retina'
from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt

from wesad_stress.features import DEFAULT_OUTPUT, CONDITION_LABELS

IMAGES = Path("../images")
SAVE_DPI = 250

features = pd.read_parquet(DEFAULT_OUTPUT)
print(f"{len(features)} windows x {features.shape[1]} columns, "
      f"{features['subject'].nunique()} subjects")
features.head()

## Class balance

The protocol is unbalanced by design — baseline and meditation run ~20 min each, stress ~10 min, amusement ~6 min — so the window counts inherit that skew. Worth knowing before modelling: a constant-baseline predictor is already a non-trivial baseline, and stratified splits or class weighting will matter.


In [ ]:
label_order = [CONDITION_LABELS[k] for k in sorted(CONDITION_LABELS)]

overall = features["label_name"].value_counts().reindex(label_order)
print("Overall window counts by condition:")
print(overall.to_string())
print(f"\nMajority-class share: {overall.max() / overall.sum():.1%}")

In [ ]:
# Per-subject breakdown — confirms every subject contributes all four classes.
per_subject = (
    features.pivot_table(
        index="subject", columns="label_name", values="window_start_s",
        aggfunc="count", fill_value=0,
    )
    .reindex(columns=label_order)
)
per_subject["total"] = per_subject.sum(axis=1)
per_subject

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
per_subject[label_order].plot(kind="bar", stacked=True, ax=ax)
ax.set_xlabel("subject")
ax.set_ylabel("window count")
ax.set_title("Window count per subject, stacked by condition")
ax.legend(title="condition", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_axisbelow(True)
ax.grid(True, axis="y", linestyle="-", linewidth=0.5, alpha=0.4)
fig.tight_layout()
fig.savefig(IMAGES / "2026-06-02-window-count-per-subject.png", dpi=SAVE_DPI, bbox_inches="tight")
plt.show()

## Feature distributions by condition

A representative slice across the five modalities. The directions to sanity-check: heart rate up under stress, HRV (RMSSD) down under stress, tonic EDA up under stress, breathing slow under meditation. These are face-validity checks on the extraction, not modelling claims.


In [ ]:
plot_features = [
    "hrv_mean_hr", "hrv_rmssd", "hrv_lf_hf",
    "eda_tonic_mean", "eda_phasic_std", "eda_scr_count",
    "emg_band_power", "resp_rate", "acc_mag_std",
]

fig, axes = plt.subplots(3, 3, figsize=(13, 11))
for ax, feat in zip(axes.ravel(), plot_features):
    data = [features.loc[features["label_name"] == lab, feat].dropna()
            for lab in label_order]
    ax.boxplot(data, labels=label_order, showfliers=False)
    ax.set_title(feat)
    ax.tick_params(axis="x", rotation=30)
    ax.set_axisbelow(True)
    ax.grid(True, axis="y", linestyle="-", linewidth=0.5, alpha=0.4)
fig.suptitle("Feature distributions across conditions (outliers hidden)", y=1.0)
fig.tight_layout()
fig.savefig(IMAGES / "2026-06-02-feature-distributions-by-condition.png", dpi=SAVE_DPI, bbox_inches="tight")
plt.show()

In [ ]:
# Median feature values per condition — the compact numeric companion to the
# boxplots above.
summary = (
    features.groupby("label_name")[plot_features]
    .median()
    .reindex(label_order)
    .round(2)
)
summary

---

This notebook demonstrated the feature table loading cleanly, every subject contributing windows across all four conditions, and the headline features separating in the physiologically expected directions — elevated heart rate and tonic EDA under stress, suppressed HRV under stress, and slowed respiration under meditation. The two saved figures back the feature-extraction entry in `CHANGELOG.md`; the table is the input to the baseline models.
